# Tradeify Risk Copilot — QLoRA + ORPO fine-tune (Google Colab, free T4)

**What this does:** teaches Qwen2.5-7B-Instruct *citation discipline* — every factual
claim carries a `[chunk_xxx]` citation, and out-of-rulebook questions get the exact
refusal string. Knowledge stays in the RAG corpus; the weights learn *behavior*.

**You need:** a Colab runtime with GPU (Runtime → Change runtime type → T4 GPU).
Expected wall time: 2–4 hours. Colab free may disconnect idle sessions — keep the
tab open. Nothing here costs money.

**Trade-off (documented):** the DeepSeek plan called for 14B. A free T4 (~15 GB VRAM)
cannot reliably fit 14B QLoRA+ORPO, so this notebook uses **7B** with batch-size 1 and
gradient accumulation. Same method, smaller model, actually runnable for free.


In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU — set Runtime → Change runtime type → GPU (T4)."
name = torch.cuda.get_device_name(0)
gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {name} ({gb:.1f} GB)")


In [ ]:
# ---- synthetic ORPO preference dataset (generated on the spot) ----
SUMMARIES = {'chunk_dll': ('Daily Loss Limit', 'daily loss limit', 'Growth accounts have a daily loss limit ($1,250 on $50K, $2,500 on $100K, $3,750 on $150K) that pauses trading for the rest of the session when hit but never fails the account; Select evaluations, Select Flex funded, and Lightning $25K have no daily loss limit.'), 'chunk_trailing_dd': ('End-of-Day Trailing Max Drawdown', 'trailing drawdown', 'Every account uses an end-of-day trailing drawdown that only moves up with the highest session close, locks permanently at $100 above the starting balance once cleared, and fails the account permanently if intraday equity touches the floor.'), 'chunk_consistency': ('Consistency Rule', 'consistency rule', 'Select evaluations cap the best day at 40% of total profit, Growth funded accounts at 35%, and Lightning uses a progressive 20%/25%/30% rule across payouts; breaking it only delays payout eligibility and never fails the account.'), 'chunk_profit_target': ('Profit Targets', 'profit target', 'Growth evaluation targets are $3,000/$6,000/$9,000 and Select evaluation targets are $2,500/$6,000/$9,000 by account size; Growth evaluations can be passed in a single day, and funded accounts have no profit target.'), 'chunk_microscalp': ('Microscalping Rule', 'microscalping rule', 'Over 50% of trades and over 50% of profit must come from positions held longer than 10 seconds; otherwise evaluation activation and payouts stay blocked until both ratios clear.'), 'chunk_session_rules': ('Session Rules', 'session rules', 'All positions must be flat by 4:59 PM ET (12:59 PM ET on early-close holidays); no overnight or weekend holds, no hedging even across accounts, a maximum of 5 funded accounts, at least one trade per week, and news trading is allowed.'), 'chunk_account_families': ('Account Families', 'account types', 'Growth is an evaluation with a soft daily loss limit and no consistency rule (35% once funded); Select evaluation has no DLL but a 40% consistency rule, with no consistency rule once funded; Lightning is instant funding with a progressive 20%/25%/30% consistency rule.'), 'chunk_payouts': ('Payouts and Profit Split', 'payouts', 'Traders keep 90% of profits (100% of the first $15,000 on eligible plans) with an advertised 60-minute payout guarantee, reaching Elite Live capital after 5 payouts.'), 'chunk_drawdown_recovery': ('Drawdown Recovery', 'drawdown recovery', 'Because the trailing floor only moves on end-of-day balances, intraday swings never raise it; protecting the session close is what protects the account, and once the floor locks the original capital cannot be trailed into.')}
QUESTION_TEMPLATES = ['What is the {title} at Tradeify?', 'Explain the {title}.', 'How does the {short} work at Tradeify?', 'Summarize the {title} for me.', 'What should a trader know about the {short}?', 'How does Tradeify handle the {short}?', 'Tell me about the {title}.', 'What are the key points of the {title}?', 'A new trader is confused about the {short}. Explain it clearly.', 'Give me the essentials on the {title}.']
RAMBLE_PREFIX = "Well, this is a really interesting question and one that a lot of traders ask me about all the time, and I think it's important to look at it from several angles before giving a definitive answer. "
import json, random
rng = random.Random(7)
ids = list(SUMMARIES)
pairs = []
for cid in ids:
    title, short, summary = SUMMARIES[cid]
    context = f'[{cid}] {title}: {summary}'
    for tmpl in QUESTION_TEMPLATES:
        q = tmpl.format(title=title, short=short)
        chosen = f'{summary} [{cid}]'
        wrong = rng.choice([i for i in ids if i != cid])
        for rej in (summary, f'{summary} [{wrong}]', RAMBLE_PREFIX + summary):
            pairs.append({'context': context, 'question': q,
                          'chosen': chosen, 'rejected': rej})
rng.shuffle(pairs)
SYSTEM = 'You are the Tradeify Risk Copilot. Answer ONLY from the context. Cite every factual claim like [chunk_dll]. If the context is insufficient, reply EXACTLY: "I don\'t have that in the Tradeify rulebook I can access."'
PROMPT = '<system>{system}</system>\n<context>{context}</context>\n<question>{question}</question>'
rows = [{'prompt': PROMPT.format(system=SYSTEM, context=p['context'], question=p['question']),
         'chosen': p['chosen'], 'rejected': p['rejected']} for p in pairs]
with open('/content/finetune_pairs.jsonl', 'w') as f:
    f.write('\n'.join(json.dumps(r) for r in rows))
print(f'wrote {len(rows)} preference pairs')
print('chosen :', rows[0]['chosen'][:110], '...')
print('rejected:', rows[0]['rejected'][:110], '...')


In [ ]:
!pip install -q torch transformers peft trl datasets bitsandbytes accelerate
print("deps installed")

In [ ]:
# ---- QLoRA + ORPO training (the long cell; go make coffee) ----
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model
from trl import ORPOConfig, ORPOTrainer

MODEL = "Qwen/Qwen2.5-7B-Instruct"
OUT = "/content/qwen7b-copilot-orpo"

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.bfloat16)
model = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=bnb, device_map="auto")
tok = AutoTokenizer.from_pretrained(MODEL)
tok.pad_token = tok.eos_token or tok.pad_token
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, LoraConfig(
    r=32, lora_alpha=64, lora_dropout=0.05, task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]))

ds = load_dataset("json", data_files="/content/finetune_pairs.jsonl")["train"]
cfg = ORPOConfig(output_dir=OUT, num_train_epochs=2,
                 per_device_train_batch_size=1, gradient_accumulation_steps=16,
                 learning_rate=8e-6, beta=0.1, max_length=1024,
                 logging_steps=10, save_steps=200, bf16=True,
                 optim="paged_adamw_8bit", remove_unused_columns=False)
ORPOTrainer(model=model, args=cfg, train_dataset=ds, tokenizer=tok).train()
model.save_pretrained(OUT)
tok.save_pretrained(OUT)
print(f"saved -> {OUT}  (download this folder before the session ends!)")


In [ ]:
# ---- quick behavioral eval: citation present? refusal exact? ----
import re, torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

BASE = "Qwen/Qwen2.5-7B-Instruct"
ADAPT = "/content/qwen7b-copilot-orpo"
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.bfloat16)
base = AutoModelForCausalLM.from_pretrained(BASE, quantization_config=bnb, device_map="auto")
tok = AutoTokenizer.from_pretrained(BASE)
model = PeftModel.from_pretrained(base, ADAPT)

CTX = "[chunk_dll] Daily Loss Limit: Growth accounts have a daily loss limit ($1,250 on $50K) that pauses trading for the session when hit but never fails the account."
SYS = "You are the Tradeify Risk Copilot. Answer ONLY from the context. Cite every factual claim like [chunk_dll]."
REFUSAL = "I don't have that in the Tradeify rulebook I can access."

def ask(question, context=CTX):
    prompt = f"<system>{SYS}</system>\n<context>{context}</context>\n<question>{question}</question>\n<answer>"
    ids = tok(prompt, return_tensors="pt").to(model.device)
    out = model.generate(**ids, max_new_tokens=120, do_sample=False)
    return tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True).strip()

tests = [
    ("What is the daily loss limit?", lambda a: bool(re.search(r"\[chunk_dll\]", a)), "citation present"),
    ("Explain the DLL.", lambda a: bool(re.search(r"\[chunk_[a-z_]+\]", a)), "any citation present"),
    ("What is the capital gains tax in Indonesia?", lambda a: a.strip() == REFUSAL, "exact refusal"),
    ("Who won the 2024 World Series?", lambda a: a.strip() == REFUSAL, "exact refusal (OOD)"),
]
ok = 0
for q, check, name in tests:
    a = ask(q, context="" if "tax" in q or "World Series" in q else CTX)
    passed = check(a)
    ok += passed
    print(f"[{'PASS' if passed else 'FAIL'}] {name}\n  Q: {q}\n  A: {a[:160]}\n")
print(f"{ok}/{len(tests)} behavioral checks passed")


## Next steps

1. **Download `/content/qwen7b-copilot-orpo`** (the LoRA adapter) before the Colab
   session ends — files vanish when the runtime is recycled.
2. Merge or load it with PEFT, then serve it (Ollama `Modelfile` / vLLM / llama.cpp)
   and point the agent's `explain` tier at it (`src/model_router.py`).
3. Re-run the eval harness against the tuned model and compare `citation_precision`
   and `refusal_accuracy` vs the base model — that delta is your interview story.
4. Better data → better model: replace the synthetic pairs with frontier-model
   generated ones (500+ pairs) and re-run this notebook unchanged.
